# Module 6: Governance Integration - DQ Meets Horizon

## Learning Objectives
- Apply object tags for data classification and SLA tracking
- Run automatic sensitive data classification
- View column lineage and object dependencies
- Understand how DQ integrates with Snowflake Horizon

## Key Concept: Quality is Governance

DQ is part of a broader governance framework: tags classify data, classification detects PII, lineage shows impact, access history proves compliance.

---

> **Role:** `CORP_DQ_ADMIN` | **Time:** ~45 minutes

> **What this does:** Sets your session context to the lab role, database, and warehouse.


In [ ]:
USE ROLE CORP_DQ_ADMIN;
USE DATABASE CORP_DWH;
USE WAREHOUSE COMPUTE_WH;

---
## 6a. Create Tags

> **Business Value:** Tags enable automated policy enforcement. When a column is tagged PII, masking policies and DQ checks automatically apply -- no manual configuration per table.

> **DQ Domain:** All (governance metadata)

In [ ]:
CREATE OR REPLACE TAG CORP_DWH.DQ.SENSITIVITY
    ALLOWED_VALUES 'PII', 'SENSITIVE', 'INTERNAL', 'PUBLIC'
    COMMENT = 'Data sensitivity classification';

CREATE OR REPLACE TAG CORP_DWH.DQ.DATA_DOMAIN
    ALLOWED_VALUES 'CUSTOMER', 'TRANSACTION', 'FINANCE', 'GOV_PORTAL'
    COMMENT = 'Business domain classification';

CREATE OR REPLACE TAG CORP_DWH.DQ.SLA_TIER
    ALLOWED_VALUES 'TIER_1', 'TIER_2', 'TIER_3'
    COMMENT = 'Data freshness SLA tier';

CREATE OR REPLACE TAG CORP_DWH.DQ.DQ_OWNER
    COMMENT = 'Team responsible for data quality';

---
## 6b. Apply Tags

> **Business Value:** SLA tier tags drive alert priority. TIER_1 failures wake up on-call at 2am; TIER_3 failures wait for next business day. Without tags, everything is urgent (so nothing is).

> **DQ Domain:** All (classification + ownership)

In [ ]:
-- Tag DIM_CUSTOMER table
ALTER TABLE CORP_DWH.GOLD.DIM_CUSTOMER SET TAG
    CORP_DWH.DQ.DATA_DOMAIN = 'CUSTOMER',
    CORP_DWH.DQ.SLA_TIER = 'TIER_1',
    CORP_DWH.DQ.DQ_OWNER = 'Data Governance Team';

-- Tag sensitive columns
ALTER TABLE CORP_DWH.GOLD.DIM_CUSTOMER MODIFY COLUMN NATIONAL_ID SET TAG CORP_DWH.DQ.SENSITIVITY = 'PII';
ALTER TABLE CORP_DWH.GOLD.DIM_CUSTOMER MODIFY COLUMN IBAN SET TAG CORP_DWH.DQ.SENSITIVITY = 'SENSITIVE';
ALTER TABLE CORP_DWH.GOLD.DIM_CUSTOMER MODIFY COLUMN EMAIL SET TAG CORP_DWH.DQ.SENSITIVITY = 'PII';
ALTER TABLE CORP_DWH.GOLD.DIM_CUSTOMER MODIFY COLUMN PHONE SET TAG CORP_DWH.DQ.SENSITIVITY = 'PII';

-- Tag FACT_TRANSACTIONS
ALTER TABLE CORP_DWH.GOLD.FACT_TRANSACTIONS SET TAG
    CORP_DWH.DQ.DATA_DOMAIN = 'TRANSACTION',
    CORP_DWH.DQ.SLA_TIER = 'TIER_2',
    CORP_DWH.DQ.DQ_OWNER = 'Finance Team';

---
## Checkpoint: Verify Tags Applied

> **What this does:** Displays all tags applied to DIM_CUSTOMER, showing which columns are tagged and their classification values (PII, SENSITIVE, etc.).


In [ ]:
SELECT
    OBJECT_NAME,
    COLUMN_NAME,
    TAG_NAME,
    TAG_VALUE
FROM TABLE(CORP_DWH.INFORMATION_SCHEMA.TAG_REFERENCES(
    'CORP_DWH.GOLD.DIM_CUSTOMER', 'TABLE'
))
ORDER BY TAG_NAME, COLUMN_NAME;

> **What this does:** Verifies your work so far. All checks should show [PASS].


In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

tags = session.sql("""
SELECT COUNT(*) AS CNT FROM TABLE(CORP_DWH.INFORMATION_SCHEMA.TAG_REFERENCES(
    'CORP_DWH.GOLD.DIM_CUSTOMER', 'TABLE'))
""").collect()[0]['CNT']

pii_tags = session.sql("""
SELECT COUNT(*) AS CNT FROM TABLE(CORP_DWH.INFORMATION_SCHEMA.TAG_REFERENCES(
    'CORP_DWH.GOLD.DIM_CUSTOMER', 'TABLE'))
WHERE TAG_VALUE = 'PII'
""").collect()[0]['CNT']

print("=" * 50)
print("CHECKPOINT: Tag Verification")
print("=" * 50)
if tags >= 7:
    print(f"  [PASS] {tags} tags applied to DIM_CUSTOMER (expected >= 7)")
else:
    print(f"  [FAIL] Only {tags} tags found (expected >= 7)")

if pii_tags == 3:
    print(f"  [PASS] {pii_tags} PII-tagged columns (NATIONAL_ID, EMAIL, PHONE)")
else:
    print(f"  [INFO] {pii_tags} PII columns found (expected 3)")
print("=" * 50)

---
## 6c. Sensitive Data Classification

> **DQ Domain:** Accuracy + Completeness (PII detection)

In [ ]:
SELECT *
FROM TABLE(SNOWFLAKE.CORE.DATA_CLASSIFICATION_SCAN('CORP_DWH.GOLD.DIM_CUSTOMER'));

---
## 6d. Object Dependencies (Lineage)

> **DQ Domain:** Consistency (impact analysis)

In [ ]:
SELECT
    REFERENCING_OBJECT_NAME,
    REFERENCING_OBJECT_DOMAIN,
    REFERENCED_OBJECT_NAME,
    REFERENCED_OBJECT_DOMAIN
FROM SNOWFLAKE.ACCOUNT_USAGE.OBJECT_DEPENDENCIES
WHERE REFERENCED_DATABASE = 'CORP_DWH' AND REFERENCED_SCHEMA = 'GOLD'
ORDER BY REFERENCED_OBJECT_NAME;

---
## Quiz: Test Your Knowledge

**Q1:** We tagged NATIONAL_ID, EMAIL, and PHONE as PII. Why is IBAN tagged as SENSITIVE instead of PII?

**Q2:** How does lineage help with DQ? Give a concrete scenario.

**Q3:** A regulator asks: "Who accessed customer PII data that had quality issues last month?" Which Snowflake features would you combine to answer this?

> **What this does:** Reveals quiz answers. Try answering first!


In [ ]:
print("""
QUIZ ANSWERS
============

Q1: PII (Personally Identifiable Information) directly identifies an individual
    (name, ID number, email, phone). IBAN identifies a bank account, which is
    SENSITIVE financial data but doesn't directly identify the person without
    additional context. The distinction matters for privacy regulations (GDPR, PDPL).

Q2: Scenario: DQ check on DIM_CUSTOMER finds 3 invalid National IDs.
    Lineage shows that V_CUSTOMER_TRANSACTIONS and V_CITY_REVENUE both depend
    on DIM_CUSTOMER. Therefore, any report using those views might show
    incorrect data. Lineage tells you the BLAST RADIUS of a quality issue.

Q3: Combine:
    1. DATA_QUALITY_MONITORING_RESULTS (find which tables had failures)
    2. TAG_REFERENCES (identify PII columns on those tables)
    3. ACCESS_HISTORY (who queried those tables during the period)
    Join these three to produce a compliance report showing:
    User X queried DIM_CUSTOMER.NATIONAL_ID on DATE when EXPECT_VALID_NATIONAL_IDS was NOT_MET.
""")

---
**Next:** Open `7_ALERTS` to close the loop with automated notifications.